In [1]:
import glob, os                      # glob finds files by pattern, os handles file names
import pandas as pd                  # tables

OWN = r"C:\Users\shiva\Downloads\HAR2\data\own"
files = sorted(glob.glob(os.path.join(OWN, "**", "*.csv"), recursive=True))   # every .csv, even in subfolders
print(len(files), "files found")

frames = []
for f in files:
    d = pd.read_csv(f)                                   # one recording
    d["file"] = os.path.basename(f)                      # remember which file each row came from
    frames.append(d)
    print(os.path.basename(f), "| rows:", len(d), "| seconds:", round(len(d) / 20))   # 20 rows = 1 second

own = pd.concat(frames, ignore_index=True)               # all recordings in one table
cols = ["ax", "ay", "az", "gx", "gy", "gz"]
print(own.groupby("label")[cols].mean().round(2))        # average of each channel, per activity
print(own.groupby("label")[cols].std().round(2))         # spread of each channel, per activity

20 files found
own_jogging_pocket_1793467391520.csv | rows: 1205 | seconds: 60
own_jogging_pocket_1793467471454.csv | rows: 1205 | seconds: 60
own_jogging_pocket_1793467547790.csv | rows: 1204 | seconds: 60
own_jogging_pocket_1793467730367.csv | rows: 2964 | seconds: 148
own_jogging_pocket_1793505570606.csv | rows: 1206 | seconds: 60
own_jogging_pocket_1793505658819.csv | rows: 1205 | seconds: 60
own_jogging_pocket_1793505736946.csv | rows: 1205 | seconds: 60
own_jogging_pocket_1793505819433.csv | rows: 1203 | seconds: 60
own_sitting_pocket_1793457470253.csv | rows: 1205 | seconds: 60
own_sitting_pocket_1793457746221.csv | rows: 5045 | seconds: 252
own_stairs_pocket_1793455552674.csv | rows: 1204 | seconds: 60
own_stairs_pocket_1793455630069.csv | rows: 1204 | seconds: 60
own_stairs_pocket_1793455708407.csv | rows: 1205 | seconds: 60
own_stairs_pocket_1793455946543.csv | rows: 1453 | seconds: 73
own_standing_pocket_1793465056046.csv | rows: 1207 | seconds: 60
own_standing_pocket_179346

In [2]:
import glob, os
import numpy as np
import pandas as pd

OWN = r"C:\Users\shiva\Downloads\HAR2\data\own"          # the same path as before
NPZ = r"C:\Users\shiva\Downloads\HAR2\data\windows.npz"              # HAR2\data\windows.npz

cols = ["ax", "ay", "az", "gx", "gy", "gz"]
names = ["Walking", "Jogging", "Stairs", "Sitting", "Standing"]   # same order as the model's labels

def clean_segments(d):
    v = d[cols].values                                            # the 6 channels as an array
    same = np.r_[False, np.all(v[1:] == v[:-1], axis=1)]          # True if a row is identical to the row before it
    run_id = np.cumsum(~same)                                     # a new id each time the values change
    run_len = np.bincount(run_id)[run_id]                         # length of the repeated run each row belongs to
    stale = run_len >= 3                                          # 3+ identical rows in a row = frozen burst
    start = stale & ~np.r_[False, stale[:-1]]                     # first row of each frozen burst
    seg = np.cumsum(start)                                        # segment number goes up after each burst
    return v[~stale], seg[~stale], int(stale.sum())               # real rows, their segment, how many were frozen

X_own, y_own = [], []
for f in sorted(glob.glob(os.path.join(OWN, "**", "*.csv"), recursive=True)):
    d = pd.read_csv(f)
    v, seg, n_stale = clean_segments(d)
    label = names.index(d["label"].iloc[0])
    n_win = 0
    for s in np.unique(seg):                                      # cut windows only inside unbroken stretches
        block = v[seg == s]
        for a in range(0, len(block) - 100 + 1, 50):              # 100 rows per window, step 50 (same as training)
            X_own.append(block[a:a + 100])
            y_own.append(label)
            n_win += 1
    print(f"{os.path.basename(f)[:30]:30s} rows={len(d):5d} frozen={n_stale:5d} windows={n_win}")

X_own = np.array(X_own, dtype=np.float32)
y_own = np.array(y_own)
print("own windows:", X_own.shape, "per class:", np.bincount(y_own, minlength=5))

# orientation-free "how hard is the phone shaking": std of the accelerometer magnitude in each window
def mag_std(X):
    return np.linalg.norm(X[:, :, :3], axis=2).std(axis=1)

w = np.load(NPZ)
Xw, yw = w["X"], w["y"]
rows = []
for k, n in enumerate(names):
    a = mag_std(Xw[yw == k])
    b = mag_std(X_own[y_own == k]) if (y_own == k).any() else np.array([np.nan])
    rows.append([n, np.median(a), np.median(b)])
print(pd.DataFrame(rows, columns=["activity", "WISDM median", "own median"]).round(2))

own_jogging_pocket_17934673915 rows= 1205 frozen=    0 windows=23
own_jogging_pocket_17934674714 rows= 1205 frozen=    0 windows=23
own_jogging_pocket_17934675477 rows= 1204 frozen=    0 windows=23
own_jogging_pocket_17934677303 rows= 2964 frozen= 1808 windows=19
own_jogging_pocket_17935055706 rows= 1206 frozen=    0 windows=23
own_jogging_pocket_17935056588 rows= 1205 frozen=    0 windows=23
own_jogging_pocket_17935057369 rows= 1205 frozen=    0 windows=23
own_jogging_pocket_17935058194 rows= 1203 frozen=    0 windows=23
own_sitting_pocket_17934574702 rows= 1205 frozen=    0 windows=23
own_sitting_pocket_17934577462 rows= 5045 frozen= 3864 windows=21
own_stairs_pocket_179345555267 rows= 1204 frozen=    0 windows=23
own_stairs_pocket_179345563006 rows= 1204 frozen=    0 windows=23
own_stairs_pocket_179345570840 rows= 1205 frozen=    0 windows=23
own_stairs_pocket_179345594654 rows= 1453 frozen=  257 windows=21
own_standing_pocket_1793465056 rows= 1207 frozen=    0 windows=23
own_standi

In [3]:
import glob, os, random
import numpy as np
import pandas as pd
import tensorflow as tf                                    # if "No module named tensorflow": run  %pip install tensorflow  first
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

OWN = r"C:\Users\shiva\Downloads\HAR2\data\own"
NPZ = r"C:\Users\shiva\Downloads\HAR2\data\windows.npz"

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

cols  = ["ax", "ay", "az", "gx", "gy", "gz"]
names = ["Walking", "Jogging", "Stairs", "Sitting", "Standing"]
# same numbers the app uses (training-set mean and std per channel)
MEAN = np.array([1.0253, -1.7554, -0.2430, -0.0112, -0.0079, -0.0020], dtype=np.float32)
STD  = np.array([6.4385, 8.1821, 4.6221, 1.2862, 1.5038, 1.0705], dtype=np.float32)
def norm(X): return (X - MEAN) / STD

# ---------- 1) your own recordings -> windows (frozen rows removed) ----------
def clean_segments(d):
    v = d[cols].values
    same = np.r_[False, np.all(v[1:] == v[:-1], axis=1)]
    run_id = np.cumsum(~same)
    run_len = np.bincount(run_id)[run_id]
    stale = run_len >= 3
    start = stale & ~np.r_[False, stale[:-1]]
    seg = np.cumsum(start)
    return v[~stale], seg[~stale]

X_own, y_own, f_own = [], [], []
for f in sorted(glob.glob(os.path.join(OWN, "**", "*.csv"), recursive=True)):
    d = pd.read_csv(f)
    v, seg = clean_segments(d)
    label = names.index(d["label"].iloc[0])
    for s in np.unique(seg):
        block = v[seg == s]
        for a in range(0, len(block) - 100 + 1, 50):
            X_own.append(block[a:a + 100]); y_own.append(label); f_own.append(os.path.basename(f))
X_own = np.array(X_own, dtype=np.float32); y_own = np.array(y_own); f_own = np.array(f_own)

# ---------- 2) hold out ONE clean recording per activity as the test set ----------
test_files = []
for k in range(5):
    files_k = sorted(set(f_own[y_own == k]))
    n = {f: int((f_own == f).sum()) for f in files_k}
    best = max(n.values())                                   # fully clean recordings have the most windows
    test_files.append([f for f in files_k if n[f] == best][-1])
is_test = np.isin(f_own, test_files)
Xo_tr, yo_tr = X_own[~is_test], y_own[~is_test]
Xo_te, yo_te = X_own[is_test],  y_own[is_test]
print("own test files:", test_files)
print("own train windows per class:", np.bincount(yo_tr, minlength=5))
print("own test  windows per class:", np.bincount(yo_te, minlength=5))

# ---------- 3) WISDM, same subject split as notebook 02 ----------
w = np.load(NPZ); Xw, yw, sw = w["X"], w["y"], w["subjects"]
shuf = np.random.default_rng(SEED).permutation(np.unique(sw))
tr_s, va_s, te_s = shuf[:35], shuf[35:42], shuf[42:]
Xw_tr, yw_tr = Xw[np.isin(sw, tr_s)], yw[np.isin(sw, tr_s)]
Xw_va, yw_va = Xw[np.isin(sw, va_s)], yw[np.isin(sw, va_s)]
Xw_te, yw_te = Xw[np.isin(sw, te_s)], yw[np.isin(sw, te_s)]

# ---------- 4) the same CNN as notebook 02 ----------
def build_cnn(input_shape=(100, 6), n_classes=5):
    inp = keras.Input(shape=input_shape)
    x = layers.Conv1D(32, 5, padding="same", use_bias=False)(inp)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv1D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x); x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def evaluate(m, X, y):
    p = m.predict(norm(X), verbose=0).argmax(axis=1)
    return accuracy_score(y, p), f1_score(y, p, average="macro"), confusion_matrix(y, p, labels=range(5))

# ---------- 5) Experiment A: WISDM only ----------
tf.random.set_seed(SEED)
mA = build_cnn()
mA.fit(norm(Xw_tr), yw_tr, validation_data=(norm(Xw_va), yw_va), epochs=60, batch_size=128, verbose=2,
       callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)])

# ---------- 6) Experiment B: WISDM + your own recordings (repeated 10x) ----------
REPEAT = 10
Xb = np.concatenate([Xw_tr] + [Xo_tr] * REPEAT)
yb = np.concatenate([yw_tr] + [yo_tr] * REPEAT)
tf.random.set_seed(SEED)
mB = build_cnn()
mB.fit(norm(Xb), yb, validation_data=(norm(Xw_va), yw_va), epochs=20, batch_size=128, verbose=2)

# ---------- 7) results ----------
rows = []
for tag, m in [("A: WISDM only", mA), ("B: WISDM + own", mB)]:
    a1, f1w, _ = evaluate(m, Xw_te, yw_te)
    a2, f1o, cm = evaluate(m, Xo_te, yo_te)
    rows.append([tag, a1, f1w, a2, f1o])
    print(f"\n{tag}: confusion matrix on YOUR held-out recordings (rows = true, columns = predicted)")
    print(pd.DataFrame(cm, index=names, columns=names))
print()
print(pd.DataFrame(rows, columns=["model", "WISDM test acc", "WISDM test F1", "own test acc", "own test F1"]).round(3))

own test files: [np.str_('own_walking_pocket_1793455472003.csv'), np.str_('own_jogging_pocket_1793505819433.csv'), np.str_('own_stairs_pocket_1793455708407.csv'), np.str_('own_sitting_pocket_1793457470253.csv'), np.str_('own_standing_pocket_1793465154388.csv')]
own train windows per class: [ 67 157  67  21  23]
own test  windows per class: [23 23 23 23 23]
Epoch 1/60
95/95 - 7s - 79ms/step - accuracy: 0.6560 - loss: 0.8998 - val_accuracy: 0.6880 - val_loss: 0.9310
Epoch 2/60
95/95 - 3s - 33ms/step - accuracy: 0.8120 - loss: 0.4774 - val_accuracy: 0.6640 - val_loss: 0.7502
Epoch 3/60
95/95 - 3s - 34ms/step - accuracy: 0.8572 - loss: 0.3634 - val_accuracy: 0.7022 - val_loss: 0.6282
Epoch 4/60
95/95 - 3s - 32ms/step - accuracy: 0.8955 - loss: 0.2902 - val_accuracy: 0.7796 - val_loss: 0.4969
Epoch 5/60
95/95 - 3s - 33ms/step - accuracy: 0.9161 - loss: 0.2394 - val_accuracy: 0.7841 - val_loss: 0.4780
Epoch 6/60
95/95 - 3s - 35ms/step - accuracy: 0.9224 - loss: 0.2168 - val_accuracy: 0.8346 

In [4]:
# ---------- merge Walking (0) and Stairs (2) into one class ----------
names4 = ["Walking+Stairs", "Jogging", "Sitting", "Standing"]
map4 = np.array([0, 1, 0, 2, 3])                # old label -> new label
def m4(y): return map4[y]

def run4(repeat, epochs):
    if repeat > 0:
        Xb = np.concatenate([Xw_tr] + [Xo_tr] * repeat)
        yb = np.concatenate([m4(yw_tr)] + [m4(yo_tr)] * repeat)
        cb = []                                  # fixed number of epochs when own data is mixed in
    else:
        Xb, yb = Xw_tr, m4(yw_tr)
        cb = [callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)]
    tf.random.set_seed(SEED)
    m = build_cnn(n_classes=4)
    m.fit(norm(Xb), yb, validation_data=(norm(Xw_va), m4(yw_va)),
          epochs=epochs, batch_size=128, verbose=0, callbacks=cb)
    return m

def ev4(m, X, y4):
    p = m.predict(norm(X), verbose=0).argmax(axis=1)
    return accuracy_score(y4, p), f1_score(y4, p, average="macro"), confusion_matrix(y4, p, labels=range(4))

rows = []
for tag, rep, ep in [("A: WISDM only", 0, 60), ("B: + own x5", 5, 20), ("B: + own x10", 10, 20)]:
    m = run4(rep, ep)
    a1, f1w, _ = ev4(m, Xw_te, m4(yw_te))
    a2, f1o, cm = ev4(m, Xo_te, m4(yo_te))
    rows.append([tag, a1, f1w, a2, f1o])
    print(f"\n{tag} - confusion matrix on YOUR held-out recordings (rows = true, columns = predicted)")
    print(pd.DataFrame(cm, index=names4, columns=names4))
print()
print(pd.DataFrame(rows, columns=["model", "WISDM test acc", "WISDM test F1", "own test acc", "own test F1"]).round(3))

# ---------- orientation check: average of ax, ay, az in each own recording ----------
chk = []
for f in sorted(set(f_own)):
    sel = f_own == f
    mu = X_own[sel][:, :, :3].mean(axis=(0, 1))
    chk.append([f[:28], names[y_own[sel][0]], *np.round(mu, 1)])
print("\naverage acceleration per recording (shows how the phone was oriented)")
print(pd.DataFrame(chk, columns=["file", "activity", "ax", "ay", "az"]).sort_values(["activity", "file"]).to_string(index=False))


A: WISDM only - confusion matrix on YOUR held-out recordings (rows = true, columns = predicted)
                Walking+Stairs  Jogging  Sitting  Standing
Walking+Stairs              46        0        0         0
Jogging                      8       15        0         0
Sitting                      0        0       23         0
Standing                     0        0        0        23

B: + own x5 - confusion matrix on YOUR held-out recordings (rows = true, columns = predicted)
                Walking+Stairs  Jogging  Sitting  Standing
Walking+Stairs              46        0        0         0
Jogging                      6       17        0         0
Sitting                      0        0       23         0
Standing                     0        0        0        23

B: + own x10 - confusion matrix on YOUR held-out recordings (rows = true, columns = predicted)
                Walking+Stairs  Jogging  Sitting  Standing
Walking+Stairs              46        0        0         0
Jogg

In [5]:
import os, glob

OWN = r"C:\Users\shiva\Downloads\HAR2\data\own"
NEW = r"C:\Users\shiva\Downloads\temp_new"

print("temp folder exists:", os.path.exists(NEW))
new_all = glob.glob(os.path.join(NEW, "**", "*.csv"), recursive=True)
print("csv files in temp folder:", len(new_all))

have = {os.path.basename(f) for f in glob.glob(os.path.join(OWN, "**", "*.csv"), recursive=True)}
print("csv files already in own folder:", len(have))

# newest recording time you already have (the number at the end of each file name)
latest = max(int(n.rsplit("_", 1)[1].split(".")[0]) for n in have)
print("newest file you already have ends in:", latest)

for f in sorted(new_all):
    n = os.path.basename(f)
    stamp = int(n.rsplit("_", 1)[1].split(".")[0])
    print(n, "| already in own:", n in have, "| newer than your latest:", stamp > latest)

temp folder exists: True
csv files in temp folder: 20
csv files already in own folder: 20
newest file you already have ends in: 1793505819433
own_jogging_pocket_1793467391520.csv | already in own: True | newer than your latest: False
own_jogging_pocket_1793467471454.csv | already in own: True | newer than your latest: False
own_jogging_pocket_1793467547790.csv | already in own: True | newer than your latest: False
own_jogging_pocket_1793467730367.csv | already in own: True | newer than your latest: False
own_jogging_pocket_1793505570606.csv | already in own: True | newer than your latest: False
own_jogging_pocket_1793505658819.csv | already in own: True | newer than your latest: False
own_jogging_pocket_1793505736946.csv | already in own: True | newer than your latest: False
own_jogging_pocket_1793505819433.csv | already in own: True | newer than your latest: False
own_sitting_pocket_1793457470253.csv | already in own: True | newer than your latest: False
own_sitting_pocket_17934577462

In [6]:
import os, glob, shutil
import pandas as pd

OWN = r"C:\Users\shiva\Downloads\HAR2\data\own"
NEW = r"C:\Users\shiva\Downloads\temp_new"

# copy only the files whose names you don't already have
have = {os.path.basename(f) for f in glob.glob(os.path.join(OWN, "**", "*.csv"), recursive=True)}
new_files = [f for f in glob.glob(os.path.join(NEW, "**", "*.csv"), recursive=True)
             if os.path.basename(f) not in have]
for f in new_files:
    shutil.copy(f, OWN)
print(len(new_files), "files copied")
print("total csv files in own:", len(glob.glob(os.path.join(OWN, "**", "*.csv"), recursive=True)))

# check every jogging recording: row count and average ax, ay, az (shows how the phone was oriented)
for f in sorted(glob.glob(os.path.join(OWN, "**", "own_jogging_*.csv"), recursive=True)):
    d = pd.read_csv(f)
    print(os.path.basename(f)[-17:], "| rows:", len(d), "| mean ax, ay, az:", d[["ax", "ay", "az"]].mean().round(1).tolist())

0 files copied
total csv files in own: 20
1793467391520.csv | rows: 1205 | mean ax, ay, az: [-2.5, 9.6, -1.1]
1793467471454.csv | rows: 1205 | mean ax, ay, az: [-1.4, 9.6, 2.1]
1793467547790.csv | rows: 1204 | mean ax, ay, az: [-0.6, -9.7, -1.9]
1793467730367.csv | rows: 2964 | mean ax, ay, az: [-0.2, -9.3, 1.0]
1793505570606.csv | rows: 1206 | mean ax, ay, az: [-0.7, 10.4, -0.1]
1793505658819.csv | rows: 1205 | mean ax, ay, az: [-0.5, -11.3, -0.1]
1793505736946.csv | rows: 1205 | mean ax, ay, az: [-0.9, 11.2, 0.7]
1793505819433.csv | rows: 1203 | mean ax, ay, az: [-0.4, -11.4, 0.7]


In [7]:
import glob, os, random
import numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

OWN    = r"C:\Users\shiva\Downloads\HAR2\data\own"
NPZ    = r"C:\Users\shiva\Downloads\HAR2\data\windows.npz"
MODELS = r"C:\Users\shiva\Downloads\HAR2\models"
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

cols  = ["ax", "ay", "az", "gx", "gy", "gz"]
names = ["Walking", "Jogging", "Stairs", "Sitting", "Standing"]     # original 5 labels
names4 = ["Walking+Stairs", "Jogging", "Sitting", "Standing"]       # merged 4 labels
map4 = np.array([0, 1, 0, 2, 3])                                    # old label -> new label
def m4(y): return map4[y]
MEAN = np.array([1.0253, -1.7554, -0.2430, -0.0112, -0.0079, -0.0020], dtype=np.float32)
STD  = np.array([6.4385, 8.1821, 4.6221, 1.2862, 1.5038, 1.0705], dtype=np.float32)
def norm(X): return (X - MEAN) / STD

TEST_FILES = {                                         # recordings the model never trains on in this cell
    "own_walking_pocket_1793455472003.csv", "own_stairs_pocket_1793455708407.csv",
    "own_sitting_pocket_1793457470253.csv", "own_standing_pocket_1793465154388.csv",
    "own_jogging_pocket_1793505570606.csv",            # jogging, phone upright
    "own_jogging_pocket_1793505658819.csv"}            # jogging, phone upside down

def clean_segments(d):                                 # removes frozen rows (3+ identical rows in a row)
    v = d[cols].values
    same = np.r_[False, np.all(v[1:] == v[:-1], axis=1)]
    run_id = np.cumsum(~same); run_len = np.bincount(run_id)[run_id]
    stale = run_len >= 3
    seg = np.cumsum(stale & ~np.r_[False, stale[:-1]])
    return v[~stale], seg[~stale]

X_own, y_own, f_own = [], [], []
for f in sorted(glob.glob(os.path.join(OWN, "**", "*.csv"), recursive=True)):
    d = pd.read_csv(f); v, seg = clean_segments(d); label = names.index(d["label"].iloc[0])
    for s in np.unique(seg):
        block = v[seg == s]
        for a in range(0, len(block) - 100 + 1, 50):
            X_own.append(block[a:a + 100]); y_own.append(label); f_own.append(os.path.basename(f))
X_own = np.array(X_own, dtype=np.float32); y_own = np.array(y_own); f_own = np.array(f_own)
assert len(X_own) > 0, "No recordings found. Check the OWN path."
is_test = np.isin(f_own, list(TEST_FILES))
Xo_tr, yo_tr, Xo_te, yo_te = X_own[~is_test], y_own[~is_test], X_own[is_test], y_own[is_test]
print("recordings:", len(set(f_own)), "| own train windows:", np.bincount(m4(yo_tr), minlength=4),
      "| own test windows:", np.bincount(m4(yo_te), minlength=4))

w = np.load(NPZ); Xw, yw, sw = w["X"], w["y"], w["subjects"]
shuf = np.random.default_rng(SEED).permutation(np.unique(sw))          # same subject split as notebook 02
tr_s, va_s, te_s = shuf[:35], shuf[35:42], shuf[42:]
Xw_tr, yw_tr = Xw[np.isin(sw, tr_s)], yw[np.isin(sw, tr_s)]
Xw_va, yw_va = Xw[np.isin(sw, va_s)], yw[np.isin(sw, va_s)]
Xw_te, yw_te = Xw[np.isin(sw, te_s)], yw[np.isin(sw, te_s)]

def build_cnn(input_shape=(100, 6), n_classes=4):      # same CNN as notebook 02
    inp = keras.Input(shape=input_shape)
    x = layers.Conv1D(32, 5, padding="same", use_bias=False)(inp)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv1D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x); x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

def ev(m, X, y4):
    p = m.predict(norm(X), verbose=0).argmax(axis=1)
    return accuracy_score(y4, p), f1_score(y4, p, average="macro"), confusion_matrix(y4, p, labels=range(4)), p

def train(repeat):
    if repeat == 0:
        Xb, yb = Xw_tr, m4(yw_tr)
        cb = [callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)]; ep = 60
    else:
        Xb = np.concatenate([Xw_tr] + [Xo_tr] * repeat); yb = np.concatenate([m4(yw_tr)] + [m4(yo_tr)] * repeat)
        cb = []; ep = 20
    tf.random.set_seed(SEED); m = build_cnn()
    m.fit(norm(Xb), yb, validation_data=(norm(Xw_va), m4(yw_va)), epochs=ep, batch_size=128, verbose=0, callbacks=cb)
    return m

rows = []
for tag, rep in [("A: WISDM only", 0), ("B: + own x10", 10)]:
    m = train(rep)
    a1, f1w, _, _ = ev(m, Xw_te, m4(yw_te))
    a2, f1o, cm, p = ev(m, Xo_te, m4(yo_te))
    rows.append([tag, a1, f1w, a2, f1o])
    print(f"\n{tag}: confusion matrix on YOUR held-out recordings (rows = true, columns = predicted)")
    print(pd.DataFrame(cm, index=names4, columns=names4))
    if rep > 0:                                        # how each held-out jogging recording did
        for f in sorted(TEST_FILES):
            if "jogging" in f:
                sel = f_own[is_test] == f
                print("  ", f[-17:], "correct:", int((p[sel] == 1).sum()), "/", int(sel.sum()))
print()
print(pd.DataFrame(rows, columns=["model", "WISDM test acc", "WISDM test F1", "own test acc", "own test F1"]).round(3))

recordings: 20 | own train windows: [134 134  21  23] | own test windows: [46 46 23 23]

A: WISDM only: confusion matrix on YOUR held-out recordings (rows = true, columns = predicted)
                Walking+Stairs  Jogging  Sitting  Standing
Walking+Stairs              46        0        0         0
Jogging                      1       45        0         0
Sitting                      0        0       23         0
Standing                     0        0        0        23

B: + own x10: confusion matrix on YOUR held-out recordings (rows = true, columns = predicted)
                Walking+Stairs  Jogging  Sitting  Standing
Walking+Stairs              46        0        0         0
Jogging                      0       46        0         0
Sitting                      0        0       23         0
Standing                     0        0        0        23
   1793505570606.csv correct: 23 / 23
   1793505658819.csv correct: 23 / 23

           model  WISDM test acc  WISDM test F1  own t

In [8]:
REPEAT = 10
Xf = np.concatenate([Xw_tr] + [X_own] * REPEAT)                # WISDM train + ALL own windows x10
yf = np.concatenate([m4(yw_tr)] + [m4(y_own)] * REPEAT)
tf.random.set_seed(SEED)
final = build_cnn()
final.fit(norm(Xf), yf, validation_data=(norm(Xw_va), m4(yw_va)), epochs=20, batch_size=128, verbose=0)
final.save(os.path.join(MODELS, "final4.keras"))
print("final model WISDM test acc (unseen people):", round(ev(final, Xw_te, m4(yw_te))[0], 3))

# ---- float16 TFLite (this one goes into the app) ----
conv = tf.lite.TFLiteConverter.from_keras_model(final)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_types = [tf.float16]
p16 = os.path.join(MODELS, "cnn4_float16.tflite")
open(p16, "wb").write(conv.convert())
print("float16:", round(os.path.getsize(p16) / 1024, 1), "KB")

# ---- float32 for comparison, and INT8 for the report ----
p32 = os.path.join(MODELS, "cnn4_float32.tflite")
open(p32, "wb").write(tf.lite.TFLiteConverter.from_keras_model(final).convert())
print("float32:", round(os.path.getsize(p32) / 1024, 1), "KB")
try:
    def rep():
        for i in np.random.RandomState(0).choice(len(Xf), 200, replace=False):
            yield [norm(Xf[i:i + 1]).astype(np.float32)]
    c8 = tf.lite.TFLiteConverter.from_keras_model(final)
    c8.optimizations = [tf.lite.Optimize.DEFAULT]; c8.representative_dataset = rep
    c8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    c8.inference_input_type = tf.int8; c8.inference_output_type = tf.int8
    p8 = os.path.join(MODELS, "cnn4_int8.tflite")
    open(p8, "wb").write(c8.convert())
    print("int8:", round(os.path.getsize(p8) / 1024, 1), "KB")
except Exception as e:
    print("INT8 conversion skipped:", e)

# ---- sanity check: does the float16 TFLite model agree with the Keras model? ----
it = tf.lite.Interpreter(model_path=p16); it.allocate_tensors()
i_in, i_out = it.get_input_details()[0]["index"], it.get_output_details()[0]["index"]
Xt = norm(np.concatenate([Xw_te[::20], X_own[::5]]))           # a sample of windows
keras_pred = final.predict(Xt, verbose=0).argmax(axis=1)
tfl_pred = []
for i in range(len(Xt)):
    it.set_tensor(i_in, Xt[i:i + 1].astype(np.float32)); it.invoke()
    tfl_pred.append(int(it.get_tensor(i_out).argmax()))
print("TFLite agrees with Keras on", round(100 * np.mean(np.array(tfl_pred) == keras_pred), 1), "% of windows")

final model WISDM test acc (unseen people): 0.895
INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmpumxv0y0g\assets


INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmpumxv0y0g\assets


Saved artifact at 'C:\Users\shiva\AppData\Local\Temp\tmpumxv0y0g'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100, 6), dtype=tf.float32, name='keras_tensor_77')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2227484725664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227473978816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227473980752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227473978112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227473978640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227474286896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471419168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471420752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471248720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471418992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471

INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmpf9w4n5v6\assets


Saved artifact at 'C:\Users\shiva\AppData\Local\Temp\tmpf9w4n5v6'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100, 6), dtype=tf.float32, name='keras_tensor_77')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2227484725664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227473978816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227473980752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227473978112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227473978640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227474286896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471419168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471420752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471248720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471418992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471

INFO:tensorflow:Assets written to: C:\Users\shiva\AppData\Local\Temp\tmp2vacjweq\assets


Saved artifact at 'C:\Users\shiva\AppData\Local\Temp\tmp2vacjweq'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 100, 6), dtype=tf.float32, name='keras_tensor_77')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2227484725664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227473978816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227473980752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227473978112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227473978640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227474286896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471419168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471420752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471248720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471418992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2227471

c:\Users\shiva\AppData\Local\Programs\Python\Python39\lib\site-packages\tensorflow\lite\python\convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
c:\Users\shiva\AppData\Local\Programs\Python\Python39\lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


int8: 21.1 KB
TFLite agrees with Keras on 100.0 % of windows
